# Bước 11 – PySpark kết nối HDFS trong JupyterLab

Notebook chạy trong container `mis5007-spark-notebook` của cụm (http://localhost:8888). Trước khi chạy:
1. Trên máy: `bash scripts/run_all.sh` (tạo `data/landing/*.csv`).
2. Trên máy: `bash scripts/11b_hdfs_yarn_cli.sh truoc` (đưa CSV thô lên HDFS bằng `hdfs dfs -put`).

**Không bịa số:** mọi con số trong báo cáo lấy từ file CSV trong `outputs/tables/`, không chép từ màn hình notebook.

## 1. Mở phiên Spark và trỏ tới HDFS

In [1]:
import os
os.environ.setdefault('HADOOP_USER_NAME', 'root')   # để ghi được vào HDFS
from pyspark.sql import SparkSession, functions as F, types as T
PROJECT = '/home/jovyan/work/mis5007_tram_y_te'
HDFS = 'hdfs://namenode:9000/mis5007'
spark = (SparkSession.builder.appName('mis5007_notebook').master('local[*]')
         .config('spark.driver.memory', '2g').getOrCreate())
spark

## 2. Đọc CSV thô từ HDFS với schema khai báo
Không để Spark tự đoán kiểu. Chế độ `PERMISSIVE` đưa dòng sai kiểu hoặc sai số cột vào cột `_corrupt_record` thay vì làm hỏng cả lần đọc.

In [2]:
schema = T.StructType([T.StructField('year', T.IntegerType()), T.StructField('row', T.IntegerType()),
                       T.StructField('col', T.IntegerType()), T.StructField('pop_raw', T.DoubleType()),
                       T.StructField('_corrupt_record', T.StringType())])
pop = (spark.read.option('header', True).option('mode', 'PERMISSIVE')
       .option('columnNameOfCorruptRecord', '_corrupt_record').schema(schema)
       .csv(f'{HDFS}/landing/pop_100m_raw.csv')).cache()
pop.printSchema()
print('Số dòng thô:', pop.count())
pop.show(5)

root
 |-- year: integer (nullable = true)
 |-- row: integer (nullable = true)
 |-- col: integer (nullable = true)
 |-- pop_raw: double (nullable = true)
 |-- _corrupt_record: string (nullable = true)

Số dòng thô: 3006720
+----+---+---+--------+---------------+
|year|row|col| pop_raw|_corrupt_record|
+----+---+---+--------+---------------+
|2015|  0|  0|-99999.0|           NULL|
|2015|  0|  1|-99999.0|           NULL|
|2015|  0|  2|-99999.0|           NULL|
|2015|  0|  3|-99999.0|           NULL|
|2015|  0|  4|-99999.0|           NULL|
+----+---+---+--------+---------------+
only showing top 5 rows



## 3. Kiểm tra: dòng nào hợp lệ, dòng nào phải cách ly?
`-99999` là mã NoData của WorldPop (pixel không có dân cư). Đây là dữ liệu lỗi thật trong nguồn, không phải lỗi tạo ra để minh hoạ.

In [3]:
(pop.withColumn('tinh_trang', F.when(F.col('_corrupt_record').isNotNull(), 'sai kiểu / sai số cột')
                              .when(F.col('pop_raw').isNull(), 'thiếu giá trị')
                              .when(F.col('pop_raw') < 0, 'giá trị âm (NoData)').otherwise('giá trị hợp lệ'))
    .groupBy('year', 'tinh_trang').count().orderBy('year', 'tinh_trang').show(30, truncate=False))

+----+-------------------+------+
|year|tinh_trang         |count |
+----+-------------------+------+
|2015|giá trị hợp lệ     |66158 |
|2015|giá trị âm (NoData)|234514|
|2016|giá trị hợp lệ     |66373 |
|2016|giá trị âm (NoData)|234299|
|2017|giá trị hợp lệ     |66581 |
|2017|giá trị âm (NoData)|234091|
|2018|giá trị hợp lệ     |66784 |
|2018|giá trị âm (NoData)|233888|
|2019|giá trị hợp lệ     |67021 |
|2019|giá trị âm (NoData)|233651|
|2020|giá trị hợp lệ     |67191 |
|2020|giá trị âm (NoData)|233481|
|2021|giá trị hợp lệ     |67497 |
|2021|giá trị âm (NoData)|233175|
|2022|giá trị hợp lệ     |67591 |
|2022|giá trị âm (NoData)|233081|
|2023|giá trị hợp lệ     |67695 |
|2023|giá trị âm (NoData)|232977|
|2024|giá trị hợp lệ     |67752 |
|2024|giá trị âm (NoData)|232920|
+----+-------------------+------+



## 4. Spark SQL trên dữ liệu đã lọc

In [4]:
pop.filter('pop_raw >= 0').createOrReplaceTempView('pop_hop_le')
spark.sql('''SELECT year, COUNT(*) AS so_pixel_co_gia_tri, ROUND(SUM(pop_raw)) AS dan_trong_hop_bao
             FROM pop_hop_le GROUP BY year ORDER BY year''').show()

+----+-------------------+-----------------+
|year|so_pixel_co_gia_tri|dan_trong_hop_bao|
+----+-------------------+-----------------+
|2015|              66158|        1365817.0|
|2016|              66373|        1382700.0|
|2017|              66581|        1400570.0|
|2018|              66784|        1417483.0|
|2019|              67021|        1432014.0|
|2020|              67191|        1445227.0|
|2021|              67497|        1458809.0|
|2022|              67591|        1470601.0|
|2023|              67695|        1480939.0|
|2024|              67752|        1490535.0|
+----+-------------------+-----------------+



## 5. Chạy toàn bộ pipeline Spark (bước 11) trên HDFS
Script đọc 6 file CSV thô từ `/mis5007/landing`, tách dòng lỗi sang `/mis5007/quarantine`, ghi bảng sạch vào `/mis5007/curated`, KPI vào `/mis5007/gold` và CSV vào `outputs/tables/`. Dừng phiên Spark của notebook trước để tránh tranh bộ nhớ.

In [5]:
spark.stop()
!cd {PROJECT} && HDFS_URI={HDFS} python scripts/11_spark_pipeline.py 2>&1 | tr '\r' '\n' | grep -E 'INFO|ERROR|Traceback'

[2026-11-16T16:03:16] INFO Spark 3.5.0, master=local[*] | landing=hdfs://namenode:9000/mis5007/landing | cac vung=hdfs://namenode:9000/mis5007
[2026-11-16T16:03:28] INFO 2. dm_56_tram    tho       56 | hop le       56 | cach ly        0
[2026-11-16T16:03:54] INFO 2. cells_dim     tho   300672 | hop le   119286 | cach ly   181386
[2026-11-16T16:04:18] INFO 2. pop_100m_raw  tho  3006720 | hop le   408264 | cach ly  2598456
[2026-11-16T16:04:23] INFO 2. age_1km_raw   tho   284160 | hop le   127236 | cach ly   156924
[2026-11-16T16:04:27] INFO 2. stations_raw  tho       65 | hop le       56 | cach ly        9
[2026-11-16T16:04:45] INFO 4. T1 bang su kien o x nam: 408224 dong (41221 o co dan x toi da 10 nam); 40 dong hop le thuoc o toan 0 bi bo
[2026-11-16T16:04:49] INFO 4. T2 khoang cach: 41221 o x 56 tram = 2308376 cap
[2026-11-16T16:04:51] INFO 2. net_dist      tho    41221 | hop le    41221 | cach ly        0
[2026-11-16T16:06:37] INFO 6. Da ghi curated / quarantine / gold tai hdfs://na

## 6. Đối soát – mọi dòng phải là ĐẠT
Dòng LỆCH → đọc `logs/11_spark_pipeline_*.log`; thường do chưa chạy lại `run_all.sh` sau khi sửa dữ liệu.

In [6]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)
pd.read_csv(f'{PROJECT}/outputs/tables/doi_soat_spark.csv')

,phep_doi_soat,spark,pandas_hoac_nguon,ket_qua,ghi_chu
0,Số dòng cách ly ghi ra vùng quarantine = tổng số dòng cách ly các nguồn,2.936775e+06,2.936775e+06,ĐẠT,hdfs://namenode:9000/mis5007/quarantine/dong_loi
1,Pixel × năm hợp lệ = dòng bảng sự kiện + dòng thuộc ô toàn 0,4.082640e+05,4.082640e+05,ĐẠT,NaN
2,Số dòng bảng sự kiện đọc lại từ vùng curated = trước khi ghi,4.082240e+05,4.082240e+05,ĐẠT,hdfs://namenode:9000/mis5007/curated/fact_dan_so_o_nam
3,"Tổng dân thô (pixel ≥ 0, mọi năm) = hợp lệ + cách ly",1.434469e+07,1.434469e+07,ĐẠT,hợp lệ 11335050; cách ly (ngoài 56 xã) 3009645
4,Tổng dân bảng sự kiện = tổng dân hợp lệ,1.133505e+07,1.133505e+07,ĐẠT,NaN
5,Số trạm hợp lệ,5.600000e+01,5.600000e+01,ĐẠT,NaN
6,"Toạ độ trạm sau chiếu UTM: Spark vs pandas/pyproj, lệch lớn nhất (m)",0.000000e+00,0.000000e+00,ĐẠT,NaN
7,Số ô lưới có dân trong 56 xã,4.122100e+04,4.122100e+04,ĐẠT,NaN
8,"Số cặp (xã, năm) khớp khoá",5.600000e+02,5.600000e+02,ĐẠT,NaN
9,Tổng pop theo xã × năm: sai số tương đối lớn nhất,1.998040e-16,0.000000e+00,ĐẠT,NaN


## 7. Dữ liệu hợp lệ và dữ liệu cách ly

In [7]:
pd.read_csv(f'{PROJECT}/outputs/tables/spark_so_dong_theo_nguon.csv')

,nguon,so_dong_tho,so_dong_hop_le,so_dong_cach_ly,ty_le_cach_ly_pct
0,dm_56_tram,56,56,0,0.000
1,cells_dim,300672,119286,181386,60.327
2,pop_100m_raw,3006720,408264,2598456,86.422
3,age_1km_raw,284160,127236,156924,55.224
4,stations_raw,65,56,9,13.846
5,net_dist,41221,41221,0,0.000


In [8]:
pd.read_csv(f'{PROJECT}/outputs/tables/spark_quarantine_tong_hop.csv')

,nguon,ly_do,so_dong
0,age_1km_raw,GIA_TRI_AM_NODATA,156924
1,cells_dim,NGOAI_56_XA,181386
2,pop_100m_raw,GIA_TRI_AM_NODATA,784596
3,pop_100m_raw,GIA_TRI_AM_NODATA;NGOAI_56_XA,1551481
4,pop_100m_raw,NGOAI_56_XA,262379
5,stations_raw,KHONG_PHAI_BAN_GHI_TRAM,8
6,stations_raw,SAI_DINH_DANG_TOA_DO,1


## 8. Bảng kết quả KPI

In [9]:
pd.read_csv(f'{PROJECT}/outputs/tables/kpi_tong_hop_spark.csv')

,chi_so,gia_tri,don_vi,loai,ghi_chu
0,"Tổng dân 56 xã/phường năm 2024 (WorldPop, ước tính)",1177774.000,người,nền,A2
1,Tổng dân 60+ năm 2024,134755.000,người,nền,A3
2,Số trạm y tế,56.000,trạm,nền,toạ độ nhập tay (A13)
3,Tỷ lệ dân trong 3 km đường chim bay tới trạm gần nhất,98.670,%,KPI chính,ngưỡng 3 km là giả định A1
4,Tỷ lệ dân trong 3 km theo mạng đường,96.050,%,KPI chính (đường đi),A6
5,Tỷ lệ dân tới trạm trong 15 phút xe máy,99.680,%,KPI chính (thời gian),A6
6,"Hệ số vòng đường trung vị (mạng đường / chim bay, ô cách trạm > 500 m)",1.368,lần,thuận tiện đường đi,A6
7,Thời gian trung vị tới trạm (xe máy),2.600,phút,thuận tiện đường đi,A6
8,Tỷ lệ dân trong 2 km,95.160,%,độ nhạy,A1
9,Tỷ lệ dân trong 5 km,99.770,%,độ nhạy,A1


In [10]:
pd.read_csv(f'{PROJECT}/outputs/tables/kpi_theo_nam_spark.csv')

,year,dan,dan_60,ty_le_trong_2km_pct,ty_le_trong_3km_pct,ty_le_trong_5km_pct,ty_le_60_trong_3km_pct,dan_tren_5km,dan_60_tren_5km,kc_trung_vi_m,so_tram,dan_binh_quan_moi_tram
0,2015,1.079220e+06,86238.869377,95.175,98.676,99.770,98.653,2487.159064,202.042818,610.1,56,19271.777236
1,2016,1.092573e+06,90608.915794,95.173,98.675,99.769,98.653,2518.494978,212.274984,610.1,56,19510.240492
2,2017,1.106759e+06,95086.982355,95.174,98.676,99.770,98.654,2550.496914,222.638506,610.2,56,19763.544927
3,2018,1.120107e+06,100089.224366,95.172,98.675,99.769,98.653,2582.332140,234.373085,610.2,56,20001.912552
4,2019,1.131569e+06,105178.593467,95.169,98.674,99.769,98.653,2610.268663,246.360696,610.4,56,20206.582748
5,2020,1.142021e+06,111041.729515,95.169,98.675,99.769,98.654,2633.382386,259.928133,610.4,56,20393.237986
6,2021,1.152760e+06,117264.200050,95.169,98.675,99.770,98.654,2657.070291,274.321977,610.5,56,20585.007775
7,2022,1.162055e+06,122231.963340,95.166,98.674,99.769,98.654,2679.466824,285.987047,610.8,56,20750.986344
8,2023,1.170211e+06,128373.247557,95.165,98.674,99.770,98.654,2697.321789,300.189945,610.9,56,20896.633822
9,2024,1.177774e+06,134755.423016,95.164,98.674,99.769,98.654,2714.982458,315.082777,610.9,56,21031.680714


In [11]:
pd.read_csv(f'{PROJECT}/outputs/tables/kpi_theo_nhom_spark.csv')

,nhom,dan,dan_60,ty_le_trong_2km_pct,ty_le_trong_3km_pct,ty_le_trong_5km_pct,ty_le_60_trong_3km_pct,dan_tren_5km,dan_60_tren_5km,kc_trung_vi_m,ty_le_trong_3km_mang_duong_pct,ty_le_trong_5km_mang_duong_pct,ty_le_toi_tram_15_phut_pct,so_xa
0,Phường,1024757.278,117162.285,97.652,99.714,99.930,99.709,719.969,83.108,559.586,98.290,99.796,99.900,45
1,Xã miền núi (địa hình),8478.883,969.450,30.486,45.359,82.019,45.359,1524.569,174.315,3385.877,32.601,52.911,76.046,2
2,Xã đồng bằng/ven đô,144537.959,16623.688,81.317,94.430,99.675,94.331,470.444,57.660,1137.976,83.911,98.990,99.538,9


## 9. Minh chứng
- HDFS NameNode: http://localhost:9871 → Utilities → Browse the file system → `/mis5007` (chụp các thư mục `landing`, `curated`, `quarantine`, `gold`).
- YARN: http://localhost:8088.
- Trên máy: `bash scripts/11b_hdfs_yarn_cli.sh sau` ghi kết quả các lệnh `hdfs dfs` và `yarn` vào `logs/`.